# Modèles de référence sur données réelles

Deux modèles d'évaluation, entraînés sur le train réel et évalués sur la validation : un gradient boosting et un petit réseau de neurones. Ce sont exactement ces modèles, avec ces réglages, qu'on entraînera plus tard sur les données synthétiques.

On les entraîne aussi sur 25, 50 et 75 % du train. Cette courbe « réel seul » servira de point de comparaison pour la courbe de mélange réel + synthétique.

Le test n'est pas chargé : il ne sert qu'à l'évaluation finale.

In [7]:
import os
import pandas as pd
import torch
from torch import nn
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import r2_score, roc_auc_score

## 1. Données

In [8]:
cal_train = pd.read_csv("data/processed/california_train.csv")
cal_val = pd.read_csv("data/processed/california_val.csv")

magic_train = pd.read_csv("data/processed/magic_train.csv")
magic_val = pd.read_csv("data/processed/magic_val.csv")

In [9]:
data = {
    "california": (cal_train, cal_val, "regression"),
    "magic": (magic_train, magic_val, "classification"),
}

## 2. Gradient boosting

`HistGradientBoosting` de scikit-learn, avec les réglages par défaut. Il n'a besoin d'aucune mise à l'échelle : un arbre ne regarde que l'ordre des valeurs. Il est entraîné directement dans la boucle de la section 4.

## 3. Réseau de neurones (PyTorch)

Deux couches cachées de 64 neurones, Adam, batchs de 256, 100 epochs. Les variables passent d'abord par une transformation par quantiles vers une loi normale, sinon les valeurs extrêmes (`AveOccup`, `AveBedrms`…) déséquilibrent l'entraînement.

La perte est l'erreur quadratique pour California, et l'entropie croisée binaire pour MAGIC. Pour MAGIC, le réseau renvoie un logit plutôt qu'une probabilité. L'AUC ne dépend que de l'ordre des prédictions, donc le résultat est le même.

Le nombre d'epochs est fixé une fois pour toutes, sans arrêt anticipé. Le même réseau sera ainsi entraîné de la même façon sur données réelles et synthétiques.

In [ ]:
EPOCHS, BATCH, LR = 100, 256, 1e-3

def fit_mlp(X, y, task, seed):
    torch.manual_seed(seed)

    # Les variables passent par une transformation par quantiles, ajustée sur les données d'entraînement
    qt = QuantileTransformer(output_distribution="normal").fit(X)
    X_t = torch.tensor(qt.transform(X), dtype=torch.float32)
    y_t = torch.tensor(y.to_numpy(), dtype=torch.float32)

    model = nn.Sequential(
        nn.Linear(X_t.shape[1], 64), 
        nn.ReLU(),
        nn.Linear(64, 64), 
        nn.ReLU(),
        nn.Linear(64, 1),
    )
    loss_fn = nn.MSELoss() if task == "regression" else nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        perm = torch.randperm(len(X_t))
        for i in range(0, len(X_t), BATCH):
            idx = perm[i:i + BATCH]
            loss = loss_fn(model(X_t[idx]).squeeze(1), y_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model, qt


def predict_mlp(model, qt, X):
    X_t = torch.tensor(qt.transform(X), dtype=torch.float32)
    with torch.no_grad():
        return model(X_t).squeeze(1).numpy()

## 4. Entraînement : 4 parts de réel × 3 graines

Pour chaque jeu, part de train (25, 50, 75, 100 %) et graine : un sous-échantillon du train, les deux modèles, un score sur la validation (R² pour California, AUC pour MAGIC).

In [12]:
rows = []
for name, (train, val, task) in data.items():
    X_val, y_val = val.drop(columns="target"), val["target"]
    metric = r2_score if task == "regression" else roc_auc_score

    for frac in [0.25, 0.5, 0.75, 1.0]:
        for seed in [0, 1, 2]:
            sub = train.sample(frac=frac, random_state=seed)
            X, y = sub.drop(columns="target"), sub["target"]

            if task == "regression":
                gbm = HistGradientBoostingRegressor(random_state=seed).fit(X, y)
                pred_gbm = gbm.predict(X_val)
            else:
                gbm = HistGradientBoostingClassifier(random_state=seed).fit(X, y)
                pred_gbm = gbm.predict_proba(X_val)[:, 1]

            mlp, qt = fit_mlp(X, y, task, seed)
            pred_mlp = predict_mlp(mlp, qt, X_val)

            rows.append({"jeu": name, "modele": "gbm", "part_reel": frac, "seed": seed, "score": metric(y_val, pred_gbm)})
            rows.append({"jeu": name, "modele": "mlp", "part_reel": frac, "seed": seed, "score": metric(y_val, pred_mlp)})
        print(name, frac, "fait")

res = pd.DataFrame(rows)

california 0.25 fait
california 0.5 fait
california 0.75 fait
california 1.0 fait
magic 0.25 fait
magic 0.5 fait
magic 0.75 fait
magic 1.0 fait


## 5. Résultats

Moyenne et écart-type sur les 3 graines. L'écart-type donne le bruit de mesure : une différence plus petite que ça entre deux générateurs ne voudra rien dire.

In [13]:
res.groupby(["jeu", "modele", "part_reel"])["score"].agg(["mean", "std"]).round(3)

mean    std
jeu        modele part_reel              
california gbm    0.25       0.809  0.001
                  0.50       0.824  0.001
                  0.75       0.830  0.001
                  1.00       0.833  0.001
           mlp    0.25       0.742  0.006
                  0.50       0.768  0.012
                  0.75       0.789  0.003
                  1.00       0.793  0.007
magic      gbm    0.25       0.924  0.002
                  0.50       0.932  0.001
                  0.75       0.937  0.001
                  1.00       0.937  0.001
           mlp    0.25       0.927  0.001
                  0.50       0.933  0.001
                  0.75       0.939  0.001
                  1.00       0.939  0.001

## 6. Sauvegarde

In [14]:
os.makedirs("results", exist_ok=True)
res.to_csv("results/baseline_val.csv", index=False)